In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================
# CHARGEMENT DES DONNÉES
# ============================================
df_train = pd.read_csv('../data/kaggle_b2_fraud_train_v3.csv')

print("="*80)
print("EXPLORATORY DATA ANALYSIS - DÉTECTION DE FRAUDE")
print("="*80)

# ============================================
# 1. VUE D'ENSEMBLE DU DATASET
# ============================================
print("\n1. VUE D'ENSEMBLE")
print("-" * 80)
print(f"Nombre de lignes: {len(df_train)}")
print(f"Nombre de colonnes: {df_train.shape[1]}")
print(f"\nTypes de données:")
print(df_train.dtypes.value_counts())

# ============================================
# 2. DISTRIBUTION DE LA TARGET (is_fraud)
# ============================================
print("\n2. DISTRIBUTION DE LA VARIABLE CIBLE")
print("-" * 80)
fraud_dist = df_train['target_is_fraud'].value_counts()
fraud_pct = df_train['target_is_fraud'].value_counts(normalize=True) * 100

print(f"Non-fraude (0): {fraud_dist[0]:,} ({fraud_pct[0]:.2f}%)")
print(f"Fraude (1):     {fraud_dist[1]:,} ({fraud_pct[1]:.2f}%)")
print(f"\nRatio déséquilibre: 1:{fraud_dist[0]/fraud_dist[1]:.1f}")

# Visualisation
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
fraud_dist.plot(kind='bar', ax=ax[0], color=['green', 'red'])
ax[0].set_title('Distribution des fraudes')
ax[0].set_xlabel('target_is_fraud')
ax[0].set_ylabel('Nombre de transactions')
ax[0].set_xticklabels(['Non-fraude', 'Fraude'], rotation=0)

fraud_pct.plot(kind='pie', ax=ax[1], autopct='%1.1f%%', colors=['green', 'red'])
ax[1].set_title('Proportion fraude/non-fraude')
ax[1].set_ylabel('')
plt.tight_layout()
plt.savefig('eda_target_distribution.png', dpi=100, bbox_inches='tight')
plt.close()

# ============================================
# 3. VALEURS MANQUANTES
# ============================================
print("\n3. ANALYSE DES VALEURS MANQUANTES")
print("-" * 80)
missing = df_train.isnull().sum()
missing_pct = (missing / len(df_train)) * 100
missing_df = pd.DataFrame({
    'Colonne': missing.index,
    'Nb_Manquants': missing.values,
    'Pourcentage': missing_pct.values
}).sort_values('Nb_Manquants', ascending=False)
missing_df = missing_df[missing_df['Nb_Manquants'] > 0]

print(f"Colonnes avec valeurs manquantes: {len(missing_df)}/{df_train.shape[1]}")
print(f"\nTop 10 colonnes avec le plus de valeurs manquantes:")
print(missing_df.head(10).to_string(index=False))

# Visualisation
if len(missing_df) > 0:
    top_missing = missing_df.head(20)
    plt.figure(figsize=(10, 6))
    plt.barh(top_missing['Colonne'], top_missing['Pourcentage'])
    plt.xlabel('Pourcentage de valeurs manquantes')
    plt.title('Top 20 colonnes avec valeurs manquantes')
    plt.tight_layout()
    plt.savefig('eda_missing_values.png', dpi=100, bbox_inches='tight')
    plt.close()

# ============================================
# 4. CORRÉLATION AVEC LA TARGET
# ============================================
print("\n4. CORRÉLATION AVEC is_fraud")
print("-" * 80)
numeric_cols = df_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
if 'is_fraud' in numeric_cols:
    numeric_cols.remove('is_fraud')

correlations = []
for col in numeric_cols:
    if df_train[col].notna().sum() > 0:
        corr = df_train[[col, 'target_is_fraud']].corr().iloc[0, 1]
        correlations.append({'Feature': col, 'Correlation': corr})

corr_df = pd.DataFrame(correlations).dropna()
corr_df['Abs_Correlation'] = corr_df['Correlation'].abs()
corr_df = corr_df.sort_values('Abs_Correlation', ascending=False)

print(f"\nTop 15 features les plus corrélées avec is_fraud:")
print(corr_df.head(15)[['Feature', 'Correlation']].to_string(index=False))

# Visualisation
top_corr = corr_df.head(20)
plt.figure(figsize=(10, 8))
colors = ['red' if x < 0 else 'green' for x in top_corr['Correlation']]
plt.barh(top_corr['Feature'], top_corr['Correlation'], color=colors)
plt.xlabel('Corrélation avec is_fraud')
plt.title('Top 20 features corrélées avec la fraude')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.savefig('eda_correlations.png', dpi=100, bbox_inches='tight')
plt.close()

# ============================================
# 5. DISTRIBUTION DES FEATURES CLÉS
# ============================================
print("\n5. DISTRIBUTION DES FEATURES CLÉS")
print("-" * 80)

# Prendre les 4 features les plus corrélées
top_features = corr_df.head(4)['Feature'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(top_features):
    if col in df_train.columns:
        df_train.boxplot(column=col, by='target_is_fraud', ax=axes[idx])
        axes[idx].set_title(f'{col} par target_is_fraud')
        axes[idx].set_xlabel('target_is_fraud')
        
        # Stats descriptives
        fraud_mean = df_train[df_train['target_is_fraud']==1][col].mean()
        non_fraud_mean = df_train[df_train['target_is_fraud']==0][col].mean()
        print(f"\n{col}:")
        print(f"  Moyenne non-fraude: {non_fraud_mean:.2f}")
        print(f"  Moyenne fraude:     {fraud_mean:.2f}")

plt.suptitle('Distribution des top features par classe', y=1.02)
plt.tight_layout()
plt.savefig('eda_feature_distributions.png', dpi=100, bbox_inches='tight')
plt.close()

# ============================================
# 6. TYPES DE COLONNES
# ============================================
print("\n6. RÉPARTITION DES TYPES DE COLONNES")
print("-" * 80)
cat_cols = df_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = df_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
bool_cols = df_train.select_dtypes(include=['bool']).columns.tolist()

print(f"Colonnes numériques: {len(num_cols)}")
print(f"Colonnes catégorielles: {len(cat_cols)}")
print(f"Colonnes booléennes: {len(bool_cols)}")

# ============================================
# 7. OUTLIERS SUR TOP FEATURES
# ============================================
print("\n7. DÉTECTION D'OUTLIERS (TOP 4 FEATURES)")
print("-" * 80)

for col in top_features[:4]:
    if col in df_train.columns and df_train[col].dtype in ['int64', 'float64']:
        Q1 = df_train[col].quantile(0.25)
        Q3 = df_train[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = df_train[(df_train[col] < lower_bound) | (df_train[col] > upper_bound)]
        print(f"\n{col}:")
        print(f"  Outliers détectés: {len(outliers)} ({len(outliers)/len(df_train)*100:.2f}%)")
        print(f"  Min: {df_train[col].min():.2f}, Max: {df_train[col].max():.2f}")
        print(f"  Q1: {Q1:.2f}, Q3: {Q3:.2f}")

# ============================================
# 8. RÉSUMÉ FINAL
# ============================================
print("\n" + "="*80)
print("RÉSUMÉ DE L'EDA")
print("="*80)
print(f"""
DATASET:
- {len(df_train):,} transactions
- {df_train.shape[1]} features
- {len(cat_cols)} catégorielles, {len(num_cols)} numériques

TARGET:
- Fraude: {fraud_pct[1]:.2f}% (DÉSÉQUILIBRÉ - ratio 1:{fraud_dist[0]/fraud_dist[1]:.1f})
- Métrique prioritaire recommandée: RECALL (capturer les fraudes)

QUALITÉ DES DONNÉES:
- {len(missing_df)} colonnes avec valeurs manquantes
- Colonnes à >90% manquants: {len(missing_df[missing_df['Pourcentage'] > 90])}

FEATURES IMPORTANTES:
- Top 3 corrélées: {', '.join(corr_df.head(3)['Feature'].tolist())}

PROCHAINES ÉTAPES RECOMMANDÉES:
1. Supprimer colonnes >90% manquants
2. Imputer colonnes <90% manquants (médiane/mode)
3. Gérer le déséquilibre (SMOTE ou class_weight)
4. Tester modèles: Random Forest, Gradient Boosting, XGBoost
5. Optimiser sur RECALL pour maximiser détection fraudes
""")

print("\nGraphiques sauvegardés:")
print("- eda_target_distribution.png")
print("- eda_missing_values.png")
print("- eda_correlations.png")
print("- eda_feature_distributions.png")

EXPLORATORY DATA ANALYSIS - DÉTECTION DE FRAUDE

1. VUE D'ENSEMBLE
--------------------------------------------------------------------------------
Nombre de lignes: 160000
Nombre de colonnes: 56

Types de données:
float64    23
object     20
int64      13
Name: count, dtype: int64

2. DISTRIBUTION DE LA VARIABLE CIBLE
--------------------------------------------------------------------------------
Non-fraude (0): 155,076 (96.92%)
Fraude (1):     4,924 (3.08%)

Ratio déséquilibre: 1:31.5

3. ANALYSE DES VALEURS MANQUANTES
--------------------------------------------------------------------------------
Colonnes avec valeurs manquantes: 14/56

Top 10 colonnes avec le plus de valeurs manquantes:
               Colonne  Nb_Manquants  Pourcentage
partner_risk_indicator        155249    97.030625
  legacy_partner_score        153736    96.085000
       secondary_email        147391    92.119375
                region         45240    28.275000
     annual_income_eur         11186     6.99125